# ABM Inspection Simulation 

## Required files (place in the same folder)
- `init_ABM_dataset.xlsx`
- `ml_inference.py`
- `final_catboost_model.cbm`
- `final_catboost_model.meta.json`


In [ ]:
!pip install -r requirements.txt >nul 2>&1

In [ ]:
import sys; sys.path.append('..')
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import traceback

from datetime import datetime, timedelta
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.time import RandomActivation
from mesa.datacollection import DataCollector
from ml_inference import predict_inspection_result 
from matplotlib.ticker import MaxNLocator
from IPython.display import Markdown, display
from ipywidgets import HBox, VBox, IntSlider, FloatSlider, Button, IntText, Play, jslink, Output, Label, ToggleButton

EXCEL_FILE = 'init_ABM_dataset.xlsx'
GRID_W, GRID_H = 90, 55
TICK_DAYS = 90
SIM_START_DATE = datetime.today().date()

In [ ]:
#Load data

def load_rows_with_date(path, n=500, sim_start_date=None):
        df = pd.read_excel(path, parse_dates=['Last Inspection date'])
        req = {'name1','cvr_number','FBO type','Product type','Operation type','FBO Size','Inspection result','Last Inspection date'}

        if n < len(df):
            df = df.sample(n, random_state=42)
        items = []
        for idx, r in df.iterrows():
            last_date = pd.to_datetime(r['Last Inspection date']).date()
            interval_days = max(0, (sim_start_date - last_date).days)
            items.append({
                'agent_id': f"{r['name1']}|{r['cvr_number']}|{idx}",  # ensure uniqueness
                'FBO type': int(r['FBO type']),
                'Product type': int(r['Product type']),
                'Operation type': int(r['Operation type']),
                'FBO Size': int(r['FBO Size']),
                'inspection_result': int(r['Inspection result']),
                'last_inspection_date': last_date,
                'inspection_interval_days': int(interval_days),
            })
        return items

In [ ]:
# FBO agents

class FBOAgent(Agent):
    def __init__(self, uid, model, a):
        super().__init__(uid, model)
        self.agent_id = a['agent_id']
        self.FBO_type = int(a['FBO type'])
        self.Product_type = int(a['Product type'])
        self.Operation_type = int(a['Operation type'])
        self.FBO_size = int(a['FBO Size'])
        self.inspection_result = int(a['inspection_result'])
        self.compliance_level = self.inspection_result
        self.actual_compliance_level = self.compliance_level
        self.former_inspection_result = self.inspection_result
        self.last_inspection_date = a['last_inspection_date']
        self.inspection_interval_days = int(a['inspection_interval_days'])
        self.color = {1:'green', 2:'yellow', 3:'orange', 4:'red'}[self.compliance_level]

    def pre_ml_tick_update(self):
        features = {
            'FBO type': self.FBO_type,
            'Product type': self.Product_type,
            'Operation type': self.Operation_type,
            'Former Inspection result': self.former_inspection_result,
            'FBO Size': self.FBO_size,
            'Inspection interval (days)': self.inspection_interval_days,
        }
        
        ch = predict_inspection_result(features)
        
        if ch not in (-1, 0, 1):
            raise ValueError(f"Unexpected ML output: {ch}")
        
        if ch == 1 and self.inspection_interval_days != 90:
            self.actual_compliance_level = min(4, self.actual_compliance_level + 0.1)
            self.compliance_level = round(self.actual_compliance_level)
            
        self.color = {1:'green', 2:'yellow', 3:'orange', 4:'red'}[self.compliance_level]

    def inspect_and_maybe_improve(self):
        self.last_inspection_date = self.model.current_date
        self.inspection_result = self.compliance_level    
        if self.compliance_level > 1:
            self.model.at_risk_inspected_this_tick += 1

        features = {
            'FBO type': self.FBO_type,
            'Product type': self.Product_type,
            'Operation type': self.Operation_type,
            'Former Inspection result': self.former_inspection_result,
            'FBO Size': self.FBO_size,
            'Inspection interval (days)': self.inspection_interval_days,
        }
        
        ch = predict_inspection_result(features)
        if ch not in (-1, 0, 1):
            raise ValueError(f"Unexpected ML output: {ch}")
        
        if ch == -1:
            self.compliance_level = max(1, self.compliance_level - 1)
            
        self.color = {1:'green',2:'yellow',3:'orange',4:'red'}[self.compliance_level]
        self.actual_compliance_level = self.compliance_level
        self.inspection_interval_days = 90
        self.former_inspection_result = self.inspection_result

In [ ]:
# Inspector agents

class InspectorAgent(Agent):
    def __init__(self, uid, model, move_range, sight, inspection_capacity):
        super().__init__(uid, model)
        self.move_range = move_range
        self.sight = sight
        self.inspection_capacity = inspection_capacity

    def step(self):
        # Move randomly (prefer empty destination to reduce overlaps)
        for _ in range(12):
            dx = random.randint(-self.move_range, self.move_range)
            dy = random.randint(-self.move_range, self.move_range)
            nx = (self.pos[0] + dx) % self.model.grid.width
            ny = (self.pos[1] + dy) % self.model.grid.height
            if self.model.grid.is_cell_empty((nx, ny)):
                break
        self.model.grid.move_agent(self, (nx, ny))

        # Find nearby FBOs
        nb = self.model.grid.get_neighbors(
            self.pos, moore=True, include_center=False, radius=self.sight
        )
        fbos = [a for a in nb if isinstance(a, FBOAgent)]
        if not fbos:
            return

        # Normalize candidate attributes to 1–4 so weights are comparable
        def scale(arr):
            a = np.array(arr, dtype=float)
            mn, mx = a.min(), a.max()
            return np.full_like(a, 2.5) if mx == mn else 1 + 3 * (a - mn) / (mx - mn)

        sizes     = scale([f.FBO_size for f in fbos])
        formers   = scale([f.former_inspection_result for f in fbos])
        types     = scale([f.FBO_type for f in fbos])
        products  = scale([f.Product_type for f in fbos])
        intervals = scale([f.inspection_interval_days for f in fbos])

        # Risk score (higher = inspect earlier)
        w = self.model.weights
        score = (
            w["size"] * sizes
            + w["former"] * formers
            + w["type"] * types
            + w["product"] * products
            + w["interval"] * intervals
        )

        # Random inspection mode if toggled or all weights are zero
        random_condition = (
            self.model.random_inspection
            or (
                w["size"] == 0
                and w["former"] == 0
                and w["type"] == 0
                and w["product"] == 0
                and w["interval"] == 0
            )
        )

        if random_condition:
            order = np.random.permutation(len(fbos))
        else:
            order = np.argsort(-score)

        # Select targets up to capacity, avoiding duplicates across inspectors
        selected = []
        for idx in order:
            f = fbos[int(idx)]
            if f not in self.model.assigned_this_tick:
                selected.append(f)
                self.model.assigned_this_tick.add(f)
            if len(selected) >= self.inspection_capacity:
                break

        # Inspect selected targets
        for f in selected:
            f.inspect_and_maybe_improve()
            self.model.inspected_ids_this_tick.append(f.agent_id)
            self.model.inspections_this_tick += 1


In [ ]:
# Inspection model

class InspectionModel(Model):
    def __init__(self, items, num_inspectors=6, width=GRID_W, height=GRID_H, start_date=SIM_START_DATE,
                 w_size=1.0, w_former=2.0, w_type=1.0, w_product=1.0, w_interval=2.0,
                 move_range=3, sight=3, inspection_capacity=4, random_inspection=False):
        super().__init__()
        self.grid = MultiGrid(width, height, torus=True)
        self.schedule = RandomActivation(self)
        self.tick_days = TICK_DAYS
        self.current_date = start_date
        self.weights = {'size': w_size, 'former': w_former, 'type': w_type, 'product': w_product, 'interval': w_interval}
        self.random_inspection = random_inspection
        self.assigned_this_tick = set()
        self.inspected_ids_this_tick = []
        self.inspections_this_tick = 0
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0
        
        # FBOs
        for a in items:
            uid = a['agent_id']
            if uid in self.schedule._agents:
                continue
            f = FBOAgent(uid, self, a)
            x, y = self.random.randrange(width), self.random.randrange(height)
            self.grid.place_agent(f, (x, y))
            self.schedule.add(f)

        # Inspectors
        def _empty_cell():    # Place inspectors on empty cells
            tries = 0
            while True:
                x, y = self.random.randrange(width), self.random.randrange(height)
                if self.grid.is_cell_empty((x, y)):
                    return x, y
                tries += 1
                if tries > 10000:     # Fallback if grid is too full
                    return x, y
                    
        for i in range(num_inspectors):
            ins = InspectorAgent(
                f'INS_{i}', self,
                move_range, sight, inspection_capacity
            )
            x, y = _empty_cell()
            self.grid.place_agent(ins, (x, y))
            self.schedule.add(ins)


        self.datacollector = DataCollector(model_reporters={
            'compliance_level_1': lambda m: sum(1 for a in m.schedule.agents if isinstance(a, FBOAgent) and a.compliance_level == 1),
            'compliance_level_2': lambda m: sum(1 for a in m.schedule.agents if isinstance(a, FBOAgent) and a.compliance_level == 2),
            'compliance_level_3': lambda m: sum(1 for a in m.schedule.agents if isinstance(a, FBOAgent) and a.compliance_level == 3),
            'compliance_level_4': lambda m: sum(1 for a in m.schedule.agents if isinstance(a, FBOAgent) and a.compliance_level == 4),
            'inspections': lambda m: m.inspections_this_tick,

            'at_risk': lambda m: m.at_risk_this_tick,
            'at_risk_inspected': lambda m: m.at_risk_inspected_this_tick,
            'caught_rate': lambda m: (
                0.0 if m.at_risk_this_tick == 0
                else 100.0 * m.at_risk_inspected_this_tick / m.at_risk_this_tick
            ),          

        })

    def step(self):

        self.current_date = self.current_date + timedelta(days=self.tick_days)

        # Reset tick-specific trackers
        self.assigned_this_tick = set()
        self.inspected_ids_this_tick = []
        self.inspections_this_tick = 0
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0
        
        # Step 1: Pre-ML update for ALL FBOs 
        for a in list(self.schedule.agents):
            if isinstance(a, FBOAgent):
                a.pre_ml_tick_update()
            
        self.at_risk_this_tick = sum(
            1 for a in self.schedule.agents
            if isinstance(a, FBOAgent) and a.compliance_level > 1
        )

        # Step 2: Inspector move, then inspect
        for a in list(self.schedule.agents):
            if isinstance(a, InspectorAgent):
                a.step()

        # Step 3: End-of-tick update 
        inspected_set = set(self.inspected_ids_this_tick)
        for a in list(self.schedule.agents):
            if isinstance(a, FBOAgent) and a.agent_id not in inspected_set:
                a.inspection_interval_days += TICK_DAYS     # +90 days

        # Step 4: Collect data
        self.datacollector.collect(self)

In [ ]:
# Draw board for display

def color_for_level(lvl):
    return {1: 'green', 2: 'yellow', 3: 'orange', 4: 'red'}.get(lvl, 'grey')

def draw_human(ax, x, y, scale=1, z=4):
    head = plt.Circle((x, y-0.2*scale), 0.2*scale, fill=True, zorder=z) 
    ax.add_patch(head)
    ax.plot([x, x], [y-0.2*scale, y+0.6*scale], zorder=z)
    ax.plot([x, x-0.3*scale], [y+0.2*scale, y+0.4*scale], zorder=z)
    ax.plot([x, x+0.3*scale], [y+0.2*scale, y+0.4*scale], zorder=z)
    ax.plot([x, x-0.2*scale], [y+0.6*scale, y+1.0*scale], zorder=z)
    ax.plot([x, x+0.2*scale], [y+0.6*scale, y+1.0*scale], zorder=z)

def snapshot_model(m):
    snap = {'W': m.grid.width, 'H': m.grid.height, 'f1': [], 'f2': [], 'f_oth': [], 'ins': []}
    for a in list(m.schedule.agents):
        if hasattr(a, 'pos') and a.pos is not None:
            x, y = a.pos
            if isinstance(a, FBOAgent):
                rec = (x, y, a.compliance_level, a.FBO_size)
                if a.FBO_type == 1:
                    snap['f1'].append(rec)
                elif a.FBO_type == 2:
                    snap['f2'].append(rec)
                else:
                    snap['f_oth'].append(rec)
            else:
                snap['ins'].append((x, y, a.sight))
    return snap

def plot_snapshot(snap):
    W, H = snap['W'], snap['H']
    fig, ax = plt.subplots(figsize=(7, 7 * H / W if W > 0 else 7))

    if snap['f1']:
        x, y, levels, sizes = zip(*snap['f1'])
        ax.scatter(x, y, s=[18 + 30*(fs-1) for fs in sizes],
                   c=[color_for_level(lv) for lv in levels], marker='o', alpha=0.9, zorder=2, label='FBO type 1')

    if snap['f2']:
        x, y, levels, sizes = zip(*snap['f2'])
        ax.scatter(x, y, s=[18 + 30*(fs-1) for fs in sizes],
                   c=[color_for_level(lv) for lv in levels], marker='s', alpha=0.9, zorder=2, label='FBO type 2')

    if snap['f_oth']:
        x, y, levels, sizes = zip(*snap['f_oth'])
        ax.scatter(x, y, s=[18 + 30*(fs-1) for fs in sizes],
                   c=[color_for_level(lv) for lv in levels], marker='o', alpha=0.7, zorder=2, label='FBO other')

    for (x, y, r) in snap['ins']:
        draw_human(ax, x, y, scale=0.8, z=4)
        circ = plt.Circle((x, y), r, fill=False, zorder=3)
        ax.add_patch(circ)

    ax.set_xlim(-0.5, W - 0.5)
    ax.set_ylim(-0.5, H - 0.5)
    ax.invert_yaxis()
    ax.set_aspect('equal', adjustable='box')
    ax.set_title('Grid (tick viewer)')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend()
    plt.show()


In [ ]:
# UI

w_random     = ToggleButton(value=False, description='Random inspections',
                            tooltip='If on, inspectors choose targets randomly up to capacity')

w_w_size     = FloatSlider(description='w_size',     min=-4.0, max=4.0, step=0.1, value=0.0, readout_format='.1f')
w_w_former   = FloatSlider(description='w_former',   min=-4.0, max=4.0, step=0.1, value=1.5, readout_format='.1f')
w_w_type     = FloatSlider(description='w_type',     min=-4.0, max=4.0, step=0.1, value=-1.0, readout_format='.1f')
w_w_product  = FloatSlider(description='w_product',  min=-4.0, max=4.0, step=0.1, value=1.0, readout_format='.1f')
w_w_interval = FloatSlider(description='w_interval', min=-4.0, max=4.0, step=0.1, value=2.0, readout_format='.1f')

w_cap   = IntSlider(description='capacity',   min=1,  max=50,   step=1, value=25)
w_move  = IntSlider(description='move_range', min=1,  max=60,   step=1, value=18)
w_sight = IntSlider(description='sight',      min=1,  max=60,   step=1, value=20)
w_n_ins = IntSlider(description='inspectors', min=0,  max=100,  step=1, value=8)

w_n_fbo = IntSlider(description='N_FBO', min=500, max=5000, step=10, value=1000)
w_ticks = IntSlider(description='ticks', min=1, max=200, step=1, value=20)

btn_init = Button(description='Initialize')
btn_step = Button(description='Step')
play     = Play(value=0, min=0, max=w_ticks.value, step=1, interval=600, description="Press play")
slider   = IntSlider(description='frame', min=0, max=w_ticks.value, step=1, value=0)
jslink((play, 'value'), (slider, 'value'))

# Outputs
out_grid  = Output()
out_plots = Output()
out_info  = Output()


def _sync_tick_max(change):
    new_max = change["new"]
    play.max   = change['new']
    slider.max = change['new']
w_ticks.observe(_sync_tick_max, names='value')

# Layout
display(
    VBox([
        HBox([w_n_fbo, w_ticks, w_n_ins]),
        HBox([w_cap, w_move, w_sight]),
        HBox([w_w_size, w_w_former, w_w_type]),
        HBox([w_w_product, w_w_interval]),
        HBox([btn_init, btn_step, play, slider]),
        w_random,
        Label("Grid"),
        out_grid,
        Label("Plots & KPIs"),
        out_plots,
        Label("Info / Logs"),
        out_info
    ])
)


with out_info:  print("Controls and outputs mounted. Click Initialize, then Step/Play.")
with out_plots: print("Plots & KPIs area ready.")
with out_grid:  print("Grid area ready.")


In [ ]:
# Live engine

m = None
snaps = []
cur_frame = 0

# print error
def safe_print_exception(prefix):
   
    import sys, traceback
    etype, e, tb = sys.exc_info()
    if etype is None:
        return
    msg = "".join(traceback.format_exception(etype, e, tb, limit=5))
    try:
        with out_info:
            out_info.clear_output(wait=True)
            print(prefix)
            print(msg)
    except Exception:
        print(prefix)
        print(msg)

# build/reset simulation
def build_model():
    global m, snaps, cur_frame
    try:
        items = load_rows_with_date(EXCEL_FILE, w_n_fbo.value, SIM_START_DATE)
        m = InspectionModel(
            items,
            num_inspectors=w_n_ins.value,
            w_size=w_w_size.value, w_former=w_w_former.value, w_type=w_w_type.value,
            w_product=w_w_product.value, w_interval=w_w_interval.value,
            move_range=w_move.value, sight=w_sight.value, inspection_capacity=w_cap.value,
            random_inspection=w_random.value,
        )
        snaps = [snapshot_model(m)]
        cur_frame = 0
        slider.max = w_ticks.value
        play.max = w_ticks.value

        # draw first frame (grid only; no plots until first step)
        with out_grid:
            out_grid.clear_output(wait=True)
            try:
                plot_snapshot(snaps[0])
            except Exception:
                safe_print_exception("Error in plot_snapshot(snaps[0])")

        with out_plots:
            out_plots.clear_output(wait=True)

        with out_info:
            out_info.clear_output(wait=True)
            print("Model initialized. Use Step or Play to advance ticks.")
            display(Markdown(
                "**Tip:** Use the **Random inspections** switch *or* set all weights "
                "(w_size, w_former, w_type, w_product, w_interval) to **0** to inspect randomly."
            ))

    except Exception:
        safe_print_exception("Error during build_model()")


# scenario label 
def _scenario_label():
    if bool(w_random.value) or all(
        v == 0 for v in [
            w_w_size.value, w_w_former.value, w_w_type.value,
            w_w_product.value, w_w_interval.value
        ]
    ):
        return "(a) Random inspections"
    weights = [
        w_w_size.value, w_w_former.value, w_w_type.value,
        w_w_product.value, w_w_interval.value
    ]
    if len(set(weights)) == 1:
        return "(b) Equal-weight risk-based"
    return "(c) Tuned risk-based"


# draw KPI table and plots
def draw_kpis_and_plots(df):
    # KPI table 
    if {"at_risk", "at_risk_inspected", "caught_rate"}.issubset(df.columns):

        df_valid = df[df["at_risk"] > 0]

        if not df_valid.empty:
            avg_caught = df_valid["caught_rate"].mean()
        else:
            avg_caught = 0.0  

        df_kpis = df[["tick", "at_risk", "at_risk_inspected", "caught_rate"]].copy()
        df_kpis["avg_caught_rate"] = avg_caught

        try:
            display(
                df_kpis.style.format({
                    "caught_rate": "{:.1f}%",
                    "avg_caught_rate": "{:.1f}%"                   
                }).set_caption(
                    "Inspection KPIs "
                    "(avg_caught_rate excludes ticks with 0 at-risk FBOs)"
                )
            )
        except Exception:
            display(df_kpis)


# figure setting
    plt.rcParams.update({
        "figure.dpi": 160,
        "savefig.dpi": 300,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "legend.fontsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
    })

    # Common colours (green → red = safer → riskier)
    level_colors = {
        "compliance_level_1": "#2ca02c",  # green
        "compliance_level_2": "#ffcc00",  # yellow
        "compliance_level_3": "#ff7f0e",  # orange
        "compliance_level_4": "#d62728",  # red
    }


    figsize_main = (5.5, 3.5)   # main compliance plot
    figsize_small = (5.5, 2.5)  # transitions plots


    if any(c.startswith("compliance_level_") for c in df.columns):
        ymax = int(df.filter(like="compliance_level_").max().max())
    else:
        ymax = w_n_fbo.value
    ymax = int(((max(ymax, w_n_fbo.value)) // 50 + 1) * 50)

    #PLOT 1: Compliance distribution 
    plt.figure(figsize=figsize_main)
    for col, color in level_colors.items():
        if col in df.columns:
            plt.plot(
                df["tick"], df[col],
                label=col.replace("compliance_level_", "Level "),
                linewidth=1.8, color=color
            )
    plt.grid(False) 
    plt.title(f"Compliance distribution after inspections  {_scenario_label()}")
    plt.xlabel("Tick")
    plt.ylabel("Number of FBOs")
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.ylim(0, ymax)
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False)
    plt.tight_layout(pad=1.5)
    plt.show()

def step_once():
    global m, snaps, cur_frame
    if m is None:
        with out_info:
            out_info.clear_output(wait=True)
            print("Model not initialized. Click Init first.")
        return
    if cur_frame >= w_ticks.value:
        return

    try:
        # Run one simulation step
        m.step()
        snaps.append(snapshot_model(m))
        cur_frame += 1

        with out_grid:
            out_grid.clear_output(wait=True)
            try:
                plot_snapshot(snaps[cur_frame])
            except Exception:
                safe_print_exception(f"Error plotting frame {cur_frame}")

        with out_plots:
            out_plots.clear_output(wait=True)
            try:
                df = (
                    m.datacollector.get_model_vars_dataframe()
                    .reset_index()
                    .rename(columns={"index": "tick"})
                )
                draw_kpis_and_plots(df)
            except Exception:
                safe_print_exception("Error building plots")

        with out_info:
            out_info.clear_output(wait=True)
            print(f"Tick {cur_frame}/{w_ticks.value} | Inspectors: {w_n_ins.value} | FBOs: {w_n_fbo.value}")
            ids_list = m.inspected_ids_this_tick if m is not None else []
            if ids_list:
                preview = ", ".join(ids_list[:50]) + (" ..." if len(ids_list) > 50 else "")
                print(f"Inspected this tick ({len(ids_list)}): {preview}")
            else:
                print("Inspected this tick: none")

    except Exception:
        safe_print_exception("Error during step_once()")


# slider handler that also refreshes plots 
def on_slider_change(change):
    try:
        target = change['new']

        while m is not None and len(snaps)-1 < target and len(snaps)-1 < w_ticks.value:

            step_once()

        if m is not None and 0 <= target < len(snaps):
            with out_grid:
                out_grid.clear_output(wait=True)
                plot_snapshot(snaps[target])

            with out_plots:
                out_plots.clear_output(wait=True)
                try:
                    df = (
                        m.datacollector.get_model_vars_dataframe()
                        .reset_index()
                        .rename(columns={"index": "tick"})
                    )
                    draw_kpis_and_plots(df)
                except Exception:
                    safe_print_exception("Error building plots (slider)")
    except Exception:
        safe_print_exception("Error in on_slider_change()")

# Connect buttons and slider to simulation functions
btn_init.on_click(lambda _: build_model())
btn_step.on_click(lambda _: step_once())
slider.observe(on_slider_change, names='value')
